# Extract and visualize a molecular encounter

Run from the repository root or `examples/`, using the environment where MartiniFuncs is installed (`pip install -e .`). No model checkpoint is needed; no energies or forces are calculated. All arrows are stored **source MD forces**, not finite-cluster or environment-only forces.

The example reads one frame directly from the multi-frame dump, diagnoses candidate geometry, assembles complete shell molecules, and displays a selected encounter. Angles and distances are diagnostics, not validated reaction-readiness thresholds. X3D views require a trusted notebook and access to the viewer JavaScript assets.

Set UNITS and TIMESTEP_PS explicitly in the setup cell. The input script is not parsed; TIMESTEP_PS is used only to label physical time. Restart the kernel after package updates.


## Units at the ASE / Psience boundary

ASE positions and all geometry diagnostics in this notebook are in **Angstrom (Å)**. Psience `Molecule.coords` and its `mode_vectors` are in **Bohr radii (a₀)**. Every ASE-to-Psience conversion here applies `UnitsData.convert("Angstroms", "BohrRadius")`: **1 Å ≈ 1.889726 a₀**. The original ASE positions remain unchanged. Converting a Psience Molecule back to ASE requires the inverse Bohr-to-Angstrom conversion.

The arrow pipeline is separate: source force **eV/Å** × display scale **Å²/eV** → arrow displacement **Å** → arrow displacement **Bohr** passed to Psience. At the default scale, 1 eV/Å is shown as 0.2 Å ≈ 0.377945 Bohr. This does **not** convert the physical forces to atomic units or turn forces into gradients. Physical force conversion to Hartree/Bohr, if needed for a future evaluator adapter, requires both energy and length conversions.


In [ ]:
from pathlib import Path
from io import StringIO
from dataclasses import asdict
import numpy as np
import pandas as pd
from IPython.display import display
from ase.io import read
import MartiniFuncs
from MartiniFuncs.io.lammps import read_lammps_frame
from MartiniFuncs.chemistry.cpd_n4t.pairs import get_all_prereactive_pairs
from MartiniFuncs.pairs.structures import (get_pair_shell_clusters, get_cluster_positions_and_forces,
                                           pair_data_to_extxyz)
from MartiniFuncs.chemistry.cpd_n4t import CpdN4tCriteria
from MartiniFuncs.chemistry.cpd_n4t.selection import assess_prereactive_pairs
from MartiniFuncs.chemistry.cpd_n4t.visualization import plot_cluster
from MartiniFuncs.structure.alignment import align_geometry

ROOT = Path.cwd().resolve()
if not (ROOT / "tests/lammpsio_tests").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "tests/lammpsio_tests").is_dir():
    raise FileNotFoundError("Run from this repository root or its examples directory")
print("Package:", MartiniFuncs.__file__)
FIXTURES = ROOT / "tests/lammpsio_tests"
DATASET = "equilibration"  # Change to "shear" for more_test_data.
if DATASET == "equilibration":
    DUMP = FIXTURES / "multiple_frames.lammpstrj"
    DATAFILE = None  # This dump already supplies elements and masses.
elif DATASET == "shear":
    DUMP = FIXTURES / "more_test_data/2_337_shear_only.lammpstrj"
    DATAFILE = FIXTURES / "more_test_data/2_337_before.data"
elif DATASET == "custom":
    directorypath = Path("C:/Users/nicho/Documents/Martini collab/uma_data/smaller_system/continued_run")
    DUMP = directorypath / "equilibrated_2p5_cont.lammpstrj"
    INPUT = directorypath / "in.2p5gpa_equil"  # Added INPUT for custom dataset
    DATAFILE = directorypath / "equilibrated_2p5_cont.data"
else:
    raise ValueError("Choose DATASET='equilibration', 'shear', or 'custom'")
UNITS = "metal"  # explicit source LAMMPS units; no input-file inference
TIMESTEP_PS = 0.0005  # ps per MD step, for the displayed frame time
TYPE_SYMBOLS = None  # optional {1: "C", 2: "H", ...} for dumps without elements
FRAME_INDEX = 0
BOND_SCALE = 1.3
SHELL_RADIUS = 3.0        # Angstrom atom-to-pair cutoff; whole molecules retained.
CRITERIA = CpdN4tCriteria()  # No numerical orientation thresholds by default.
SELECTED_CANDIDATE = 1    # Index in this frame's candidate list, not a persistent ID.
FORCE_SCALE = 0.2         # Angstrom arrow length per eV/Angstrom of force.


Choose `DATASET = "shear"` above to use `more_test_data`. Its dump has type IDs but no element/mass columns, so `DATAFILE` supplies species and MD masses. Keep nonnegative frame indices near the beginning for a quick check: this dump is about 154 GB, and negative indices require scanning to the end. No splitting or full-trajectory loading is needed.


## Read synchronized geometry and vectors
`frame.columns` retains raw dump units; `lammps_forces` is normalized to eV/Angstrom. The physical input says `metal`. Its historical model comment is not configuration: the confirmed MD model is uma-s-1p1/omol.

### Mathematics: frames, time, units and molecular identity
Let $f$ be the zero-based source frame, $s_f$ its LAMMPS timestep, and $h=$ `TIMESTEP_PS` (ps per MD step). The displayed time is
$$t_f=s_fh.$$
`FRAME_INDEX` / `FRAME_INDICES` select $f$, not $s_f$. Where present, `FRAME_STRIDE` $q$ retains $f\bmod q=0$; `MAX_FRAMES` $F$ reads only $0\le f<F$. `STARTFRAME`, `ENDFRAME` or each `WINDOWS` entry $(a,b)$ keeps retained frames with $a\le f<b$; `None` means the available end. The lag spacing is $\Delta t=t_{j+1}-t_j$ between **retained** samples, not $h$. Paths and write/display switches do not enter these equations.

`UNITS` controls conversion on reading. In these workflows coordinates are Å, masses are atomic mass units, source forces are eV/Å, and reported velocities are Å/ps. `TYPE_SYMBOLS` sets the type-to-element mapping used to infer atomic properties when needed; an incorrect mapping can change masses and bond radii.

Where `BOND_SCALE` $s$ is used (default 1.3), atoms are connected when their periodic-image separation satisfies
$$\|\mathbf r_j+\mathbf nH-\mathbf r_i\|\le s(r_i^{\rm cov}+r_j^{\rm cov}).$$
$H$ contains cell vectors as rows, $\mathbf n$ is an integer image shift, and $r^{\rm cov}$ is the element's covalent radius (Å). Molecules are connected components of this inferred graph; increasing $s$ can merge close molecules. Persistent atom IDs are labels, not coordinate-array row indices.


In [ ]:
frame = read_lammps_frame(DUMP, index=FRAME_INDEX, units=UNITS,
                          input_datafile=DATAFILE, type_symbols=TYPE_SYMBOLS)
print(f"Frame {frame.index}, timestep {frame.timestep}, {len(frame.atoms)} atoms")
print("Cell (Angstrom):", frame.atoms.cell.array)
print("PBC:", frame.atoms.pbc, "Force units:", frame.atoms.info["force_units"])

if not np.isfinite(TIMESTEP_PS) or TIMESTEP_PS <= 0:
    raise ValueError("TIMESTEP_PS must be positive ps per MD step")
print(f"Frame time: {frame.timestep * TIMESTEP_PS:g} ps")


### Mathematics: pair screening, orientation and shell selection
`REACTIVE_RADIUS` $R$ (default 3.25 Å) screens the **whole CPD COM** against the mass-weighted reactive C=C center:
$$\min_{\mathbf n\in\mathbb Z^3}\|\mathbf R_{\rm CPD}+\mathbf nH-\mathbf R_{\rm CC}\|<R.$$
This is not a forming-bond distance. `MANUAL_SHIFT` / returned `shift` translates CPD by $\mathbf nH$; manual pair selection bypasses automatic screening.

Let $\mathbf a_1,\mathbf a_2$ be CPD terminal carbons and $\mathbf b_1,\mathbf b_2$ the dienophile C=C carbons in the jointly placed geometry. For the two matchings $\pi$ compute $d_i=\|\mathbf a_i-\mathbf b_{\pi(i)}\|$. Choose the smallest lexicographic tuple $(\max(d_1,d_2),d_1+d_2,\text{bond IDs})$. `forming_bonds` records those proposed atom pairs; its local variants change indices, not geometry.

Define $\mathbf u=\mathbf a_2-\mathbf a_1$, $\mathbf v=\mathbf b_2-\mathbf b_1$, $\mathbf s=(\mathbf b_1+\mathbf b_2-\mathbf a_1-\mathbf a_2)/2$. For each four-carbon reactive plane, center its coordinates into $X$ and take the last right singular vector of $X=U\Sigma V^T$ as its unit normal $\mathbf n$. Then
$$A(\mathbf p,\mathbf q)=\frac{180}{\pi}\arccos\!\left(\frac{|\mathbf p\cdot\mathbf q|}{\|\mathbf p\|\|\mathbf q\|}\right),$$
$$\theta_{\rm axis}=A(\mathbf u,\mathbf v),\quad\theta_{\rm plane}=A(\mathbf n_{\rm CPD},\mathbf n_{\rm MAL}),\quad\theta_{\rm approach}=A(\mathbf n_{\rm CPD},\mathbf s).$$
The implementation clips the cosine to [0,1]; these unoriented angles lie in [0°,90°]. Degenerate geometry is rejected. `CRITERIA` / `PAIR_CRITERIA` compare $\max d_i$, $|d_1-d_2|$, and each angle with their corresponding `max_*` fields (Å or degrees). Equality passes; `None` disables that numerical test. They do not establish a transition state or stereochemistry.

`SHELL_RADIUS` $R_s$ retains a **whole neighboring molecule** if an atom in a suitable periodic image lies within $R_s$ of any pair atom:
$$\min_{i\in\mathrm{shell\ molecule},\,j\in\mathrm{pair},\,\mathbf n}\|\mathbf r_i+\mathbf nH-\mathbf r_j\|\le R_s.$$
Pair force rows are copied by persistent ID without rescaling or recomputation. `SELECTED_CANDIDATE` selects a table row, not an atom ID.


In [ ]:
# forming_bonds_local: extracted pair rows (CPD then dienophile).
# forming_bonds_frame_local: rows in frame.atoms.
pairs, shifts = get_all_prereactive_pairs(frame.atoms, scale=BOND_SCALE)
assessments = assess_prereactive_pairs(frame.atoms, pairs, shifts,
    criteria=CRITERIA, scale=BOND_SCALE)
rows = []
for assessment in assessments:
    row = {"candidate": assessment.pair_index, "accepted": assessment.accepted,
           "reasons": "; ".join(assessment.reasons)}
    if assessment.geometry is not None:
        row.update(asdict(assessment.geometry))
    rows.append(row)
display(pd.DataFrame(rows))
if not assessments:
    raise ValueError("No candidates in this frame; choose another frame")
if not 0 <= SELECTED_CANDIDATE < len(assessments):
    raise ValueError("SELECTED_CANDIDATE is outside this frame's candidate list")
selected = assessments[SELECTED_CANDIDATE]
print("Selected candidate accepted:", selected.accepted, selected.reasons)


## Extract and inspect
Selection here is for inspection, including rejected candidates. It does not silently apply acceptance as a filter. Shell extraction uses `SHELL_RADIUS` and includes complete molecules; some atoms therefore lie beyond the nominal radius. The Extended XYZ round-trip below is in memory and writes no files.

In [20]:
clusters = get_pair_shell_clusters(frame.atoms, [pairs[SELECTED_CANDIDATE]],
    [shifts[SELECTED_CANDIDATE]], scale=BOND_SCALE, shell_radius=SHELL_RADIUS)
records = get_cluster_positions_and_forces(frame.atoms, frame, clusters)
buffer = StringIO()
pair_data_to_extxyz(records, buffer)
buffer.seek(0)
cluster = read(buffer, format="extxyz")
roles = cluster.arrays["pair_role"]
pair = cluster[roles != "shell"]
print("Cluster atoms:", len(cluster), "Pair atoms:", len(pair))
display(pd.DataFrame({"local_row": np.arange(len(pair)),
    "global_index": pair.arrays["global_index"],
    "lammps_id": pair.arrays["global_index"] + cluster.info["id_offset"],
    "role": pair.arrays["pair_role"], "element": pair.get_chemical_symbols()}))


Cluster atoms: 363 Pair atoms: 34


,local_row,global_index,lammps_id,role,element
0,0,1087,1088,CPD,C
1,1,1091,1092,CPD,C
2,2,1088,1089,CPD,C
3,3,1096,1097,CPD,H
4,4,1090,1091,CPD,C
5,5,1097,1098,CPD,H
6,6,1094,1095,CPD,H
7,7,1089,1090,CPD,C
8,8,1092,1093,CPD,H
9,9,1093,1094,CPD,H


Green: CPD; orange: dienophile; gray: shell. Bonds are inferred and displayed by Psience `Molecule` itself. The extraction graph is not substituted for its bonds. Suppress bonds only for full simulation-cell views (`bonds=False`). For optional guessed reaction-forming bonds, append local-index entries to the display molecule's `bonds` list before plotting; these are visual annotations, not evidence of a reaction. Viewer numbering is local; the table above provides persistent IDs.

### Mathematics: alignment and visualization units
Alignment matches persistent IDs and minimizes the equal-weight objective
$$\mathrm{RMSD}=\sqrt{\frac1N\sum_i\|\mathbf r_iQ+\mathbf t-\mathbf r_i^{\rm ref}\|^2}.$$
Centered coordinates give $X^TY=U\Sigma V^T$, $Q=U\,\mathrm{diag}(1,1,\det(UV^T))V^T$, $\mathbf t=\bar{\mathbf r}^{\rm ref}-\bar{\mathbf r}Q$. Coordinates transform as $\mathbf r'=\mathbf rQ+\mathbf t$; vectors as $\mathbf F'=\mathbf FQ$, $\mathbf v'=\mathbf vQ$ (no translation). RMSD is Å. A supplied reference changes orientation, not physical geometry.

Psience coordinates are Bohr: $\mathbf r_{\rm Bohr}=\mathbf r_{\mathrm{Angstrom}}/a_0$, $a_0\approx0.529177$ Å. The ASE-to-Psience conversion performs this step. A force arrow has displacement $\Delta\mathbf r_{\mathrm{Angstrom}}=\lambda\mathbf F$ with `FORCE_SCALE` / `force_scale` $\lambda$ in Å²/eV; arrow endpoints are converted to Bohr too. It is a display scale, not a force modification. `plot_pair_geometry` stores the Psience molecule in Bohr but explicitly plots it back in Å: its `disk_radius`, disk centers and line endpoints are all in that Å figure frame. Disks use fitted plane normals and site midpoints, with a line along $\mathbf s$. `transparency` only affects appearance. No bond-inference behavior is changed by these explanations.


In [21]:
display(plot_cluster(cluster))

In [16]:
import MartiniFuncs.io.psience as cio
mol = cio.ase_atoms_to_molecule(cluster)
mol.plot()

## Source force arrows
Red arrows use a common scale without normalization. With `FORCE_SCALE=0.2`, a force of 1 eV/Angstrom is drawn as a 0.2 Angstrom arrow. These forces describe the original periodic snapshot, **not** the isolated pair shown below. They have not yet been reproduced with the local UMA checkpoint.

In [ ]:
display(plot_cluster(pair, forces=True, force_scale=FORCE_SCALE))
if selected.geometry is not None:
    print("Forming bonds (global IDs):", selected.geometry.forming_bonds)
    print("Forming bonds (pair-local indices):", selected.geometry.forming_bonds_local)
    print("Forming distances (Angstrom):", selected.geometry.forming_distances)
    print("Angles (degrees):", selected.geometry.axis_angle,
          selected.geometry.plane_angle, selected.geometry.approach_angle)


## Check alignment without corrupting vectors
This controlled example rotates/translates a copy, then recovers the original placement by persistent atom IDs despite reversed reference row order. Real overlays require known atom correspondence; similarly named atoms in unrelated reference structures are not sufficient. Full periodic frames must first be extracted as finite, unwrapped clusters.

In [ ]:
rotated = pair.copy()
rotation = np.array([[0., -1., 0.], [1., 0., 0.], [0., 0., 1.]])
rotated.positions = pair.positions @ rotation + [2., -1., 3.]
for name in ("lammps_forces", "velocities", "momenta"):
    if name in rotated.arrays:
        rotated.arrays[name] = pair.arrays[name] @ rotation
aligned, rmsd, transform = align_geometry(rotated, pair[::-1])
np.testing.assert_allclose(aligned.positions, pair.positions, atol=1e-10)
np.testing.assert_allclose(aligned.arrays["lammps_forces"],
                           pair.arrays["lammps_forces"], atol=1e-10)
print("Recovered RMSD (Angstrom):", rmsd)


## Next checks
Change `FRAME_INDEX` to inspect another frame. Try optional `CpdN4tCriteria` thresholds only as exploratory choices, retaining rejection reasons. Use `iter_lammps_frames` for sequential analysis rather than repeatedly scanning or loading the entire trajectory.

For numerical reproduction, use `MartiniFuncs.workflows.force_validation.validate_forces` on the **full frame**, with your local uma-s-1p1 checkpoint, actual system charge/spin, and an explicit tolerance. This notebook does not generate a TS or validate a reaction barrier. No checkpoint is distributed with the package.

## Reactive planes and site separation
Green/orange disks show the CPD/dienophile reactive-plane orientations. The dashed blue line connects the CPD termini midpoint to the reactive C=C midpoint. Disk radius is in angstrom; the planes use the same carbon sites as the angle assessment.


In [ ]:
from MartiniFuncs.chemistry.cpd_n4t.visualization import plot_pair_geometry
display(plot_pair_geometry(pair, disk_radius=2.0, transparency=0.5, scale=BOND_SCALE))
